<!--
Copyright 2026 Yaroslav Mariukha
SPDX-License-Identifier: Apache-2.0
-->


# Intel DMA BFM

This tutorial models Intel read and write DMA components through their Avalon-ST command, response, and data interfaces. Descriptor and memory examples are pure Python; the complete BFM requires cocotb.

## 1. Model architecture

```text
Read path:  DUT rdma_cmd → IntelDMABFM → din data + rdma_resp → DUT
Write path: DUT wdma_cmd + dout data → IntelDMABFM → wdma_resp → DUT
                                      │
                                      ▼
                               SparseByteMemory
```

The model is a black-box behavioral replacement for DMA IP. It does not simulate internal FIFOs or Avalon-MM transactions; it reproduces the externally visible streaming behavior.

## 2. Memory, descriptors, and address regions

`SparseByteMemory` is byte-addressed and returns zero for unwritten locations. `ReadDMADescriptor.decode()` and `WriteDMADescriptor.decode()` unpack raw Intel command words into immutable dataclasses. `DMAAddressRegion` describes an allowed half-open interval `[start, end)` and is used by `IntelDMACommandMonitor` to reject out-of-range transfers.

In [1]:
from fpga_verification.sim.bfms import (
    DMAAddressRegion,
    ReadDMADescriptor,
    SparseByteMemory,
    WriteDMADescriptor,
)

memory = SparseByteMemory()
memory.write(0x1000, bytes([1, 2, 3, 4]))
print(memory.read(0x0FFE, 8))

region = DMAAddressRegion("input", start=0x1000, size=0x100)
print(region, region.contains(0x1080, 16), region.contains(0x10F8, 16))

read_word = 0x1000 | (16 << 32) | (3 << 64) | (1 << 72) | (1 << 73)
write_word = 0x8000 | (16 << 32) | (1 << 64)
print(ReadDMADescriptor.decode(read_word))
print(WriteDMADescriptor.decode(write_word))

b'\x00\x00\x01\x02\x03\x04\x00\x00'
input[0x1000..0x1100) True False
ReadDMADescriptor(address=4096, length=16, channel=3, generate_sop=True, generate_eop=True, stop=False, reset=False)
WriteDMADescriptor(address=32768, length=16, end_on_eop=True, stop=False, reset=False)


In [2]:
import cocotb
from cocotb.clock import Clock
from cocotb.triggers import RisingEdge

from fpga_verification.sim.bfms.intel_dma import (
    DMAAddressRegion,
    IntelDMABFM,
    IntelDMACommandMonitor,
    ReadDMADescriptor,
    SparseByteMemory,
    WriteDMADescriptor,
)
from cocotbext.avalon import AvalonSTBus


@cocotb.test()
async def dma_test(dut):
    cocotb.start_soon(Clock(dut.clk, 10, units="ns").start())

    memory = SparseByteMemory()
    memory.write(0x1000, bytes([1, 2, 3, 4]))

    dma = IntelDMABFM(
        dut,
        clock=dut.clk,
        reset=dut.reset,
        memory=memory,
        mode="full",
    ).start()

    command_monitor = IntelDMACommandMonitor(
        clock=dut.clk,
        reset=dut.reset,
        rdma_cmd_bus=AvalonSTBus.from_prefix(dut, "rdma_cmd"),
        wdma_cmd_bus=AvalonSTBus.from_prefix(dut, "wdma_cmd"),
        read_address_regions=[DMAAddressRegion("input", 0x1000, 0x4000)],
        write_address_regions=[DMAAddressRegion("output", 0x8000, 0x4000)],
    ).start()

    dut.reset.value = 1
    await RisingEdge(dut.clk)
    dut.reset.value = 0

    # Drive DUT-specific control here.
    written = memory.read(0x8000, 16)
    print(written)
    print(command_monitor.read_descriptors)

    dma.stop()
    command_monitor.stop()

## 3. IntelDMABFM

`mode` selects `full`, `read`, or `write` operation. Standard DUT prefixes are `rdma_cmd`, `rdma_resp`, `wdma_cmd`, `wdma_resp`, `din`, and `dout`; explicit `AvalonSTBus` objects may be passed when names differ.

On a normal read descriptor, the BFM reads bytes from memory, sends a packet on `din` with the requested channel, and schedules a completion response. Read lengths must align to the exported `din` beat because that interface has no `empty` signal in this model.

On a write descriptor, the BFM receives enough `dout` beats, truncates the final collected data to the requested byte length, commits it to memory, and returns a completion response. `read_response_delay_cycles` and `write_response_delay_cycles` control completion timing.

## 4. IntelDMACommandMonitor

The command monitor observes descriptor streams without driving them. It appends decoded commands to `read_descriptors` and `write_descriptors`. When allowed regions are configured, every ordinary descriptor must fit completely inside at least one matching region. Stop/reset control descriptors are decoded but do not perform an address check.

Pass either command buses or pre-built `AvalonSTMonitor` instances. Calling `start()` launches the read and write monitor tasks; `stop()` cancels owned tasks and monitors.

## 5. Controlling timing and cleanup

Use `dma.din_source.pause` to delay data produced by the read model and `dma.dout_sink.pause` to apply backpressure to data written by the DUT. The public `read_commands`, `write_commands`, `read_responses`, and `write_responses` queues can synchronize a test with accepted and completed descriptors.

Always call `stop()` in teardown or a `finally` block. It cancels the model tasks and all internally owned Avalon-ST sources and sinks. The BFM validates negative response delays, non-byte-aligned data widths, malformed packet flags, unaligned read sizes, and descriptor address regions with contextual errors.